# Финальное решение - 158 внутри-куковых фич + 4 frequency cross-cookie

## Подход

- **158 внутри-куковых фич** (timing, behavior, pointer, ua, event_mix, platform, search, diversity, meta, bot_flags)
- **4 frequency-only cross-cookie фичи** (ua_count, ua_log_count, item_count, item_log_count) —
  считаются по train+test без использования лейблов
- **НЕ используем:** ua_bot_rate / item_bot_rate_smooth — они дают утечку через train-лейблы
  (time-val P@R0.7=0.95, hidden test=0.41)
- **Валидация:** 5-fold StratifiedKFold (random_state=42)
- **Модели:** LightGBM + CatBoost, ансамбль через rank-mean
- **Финальный скор на hidden test: P@R0.7 = 0.82188**

In [1]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
from features import build_xy, events_in_window
from metric import precision_at_recall
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

SEED = 42

train = pd.read_csv('../data/train.csv',
    parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('../data/test.csv',
    parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('../data/events.csv.gz', parse_dates=['event_ts'])

# Фильтруем события по окну наблюдения [window_start, window_end)
ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)

# build_xy: строит все фичи + frequency cross-cookie фичи
# ВАЖНО: утекшие ua_bot_rate / item_bot_rate_smooth УДАЛЕНЫ из cross_cookie_feats
print('Building features...')
Xtr, Xte, ytr = build_xy(ev_tr, ev_te, train, test)

# Выравниваем колонки train/test
common = sorted([c for c in Xtr.columns if c in Xte.columns and c != 'cookie_id'])
Xtr = Xtr[['cookie_id'] + common].fillna(0)
Xte = Xte[['cookie_id'] + common].fillna(0)
ytr = train.set_index('cookie_id').loc[Xtr.cookie_id, 'target'].values
print(f'train: {len(train)} | test: {len(test)}')
print(f'Features: {len(common)}, Xtr: {Xtr.shape}, Xte: {Xte.shape}, bot rate: {ytr.mean():.4f}')

Building features...
train: 11091 | test: 4909
Features: 162, Xtr: (11091, 163), Xte: (4909, 163), bot rate: 0.0811


In [2]:
cols = [c for c in Xtr.columns if c != 'cookie_id']
X = Xtr[cols].values
X_test = Xte[cols].values

def rank01(s):
    return pd.Series(s).rank(method='average').values / (len(s) + 1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# 5-fold CV: LGBM + CatBoost
print('Training 5-fold CV...')
oof_l = np.zeros(len(X))
oof_c = np.zeros(len(X))
p_test_l = np.zeros(len(X_test))
p_test_c = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, ytr)):
    # LGBM: heavy regularization
    lgbm = LGBMClassifier(
        n_estimators=1000, learning_rate=0.02, num_leaves=31,
        min_child_samples=40, subsample=0.65, colsample_bytree=0.45,
        reg_lambda=10.0, max_depth=4, random_state=SEED + fold, verbose=-1
    )
    lgbm.fit(X[tr_idx], ytr[tr_idx])
    oof_l[va_idx] = lgbm.predict_proba(X[va_idx])[:, 1]
    p_test_l += lgbm.predict_proba(X_test)[:, 1] / 5

    # CatBoost
    cb = CatBoostClassifier(
        iterations=600, learning_rate=0.04, depth=4,
        l2_leaf_reg=6.0, random_seed=SEED + fold,
        verbose=False, allow_writing_files=False
    )
    cb.fit(X[tr_idx], ytr[tr_idx])
    oof_c[va_idx] = cb.predict_proba(X[va_idx])[:, 1]
    p_test_c += cb.predict_proba(X_test)[:, 1] / 5

    print(f'  Fold {fold+1}/5 done')

print(f'\nLGBM  OOF P@R0.7: {precision_at_recall(ytr, oof_l):.4f}')
print(f'CatBoost OOF P@R0.7: {precision_at_recall(ytr, oof_c):.4f}')

Training 5-fold CV...
  Fold 1/5 done
  Fold 2/5 done
  Fold 3/5 done
  Fold 4/5 done
  Fold 5/5 done

LGBM  OOF P@R0.7: 0.7915
CatBoost OOF P@R0.7: 0.7664


In [ ]:
# Подбираем лучший blend вес

print('Blend weight search:')
best_w, best_score = 0.5, 0
for w in np.arange(0.3, 0.75, 0.05):
    blend = w * rank01(oof_l) + (1 - w) * rank01(oof_c)
    s = precision_at_recall(ytr, blend)
    print(f'  w_l={w:.2f}: P@R0.7 = {s:.4f}')
    if s > best_score:
        best_score = s
        best_w = w

print(f'\nBest: w_l={best_w:.2f}, P@R0.7={best_score:.4f}')

Blend weight search:
  w_l=0.30: P@R0.7 = 0.7915
  w_l=0.35: P@R0.7 = 0.7915
  w_l=0.40: P@R0.7 = 0.7915
  w_l=0.45: P@R0.7 = 0.7925
  w_l=0.50: P@R0.7 = 0.7945
  w_l=0.55: P@R0.7 = 0.7955
  w_l=0.60: P@R0.7 = 0.7975
  w_l=0.65: P@R0.7 = 0.7950
  w_l=0.70: P@R0.7 = 0.7955

Best: w_l=0.60, P@R0.7=0.7975


In [6]:
# Финальный submission
test_blend = best_w * rank01(p_test_l) + (1 - best_w) * rank01(p_test_c)

os.makedirs('../output', exist_ok=True)
sub = pd.DataFrame({'cookie_id': Xte['cookie_id'].values, 'score': test_blend})
sub.to_csv('../output/submission.csv', index=False)

# sanity checks
sample = pd.read_csv('../sample_submission.csv')
assert sub.shape == sample.shape
assert list(sub.columns) == list(sample.columns)
assert set(sub.cookie_id) == set(sample.cookie_id)
assert sub.cookie_id.is_unique
assert sub.score.between(0, 1).all()

print(f'OK: {len(sub)} строк')
print(f'Score range: [{sub.score.min():.4f}, {sub.score.max():.4f}]')
print(f'Score mean: {sub.score.mean():.4f}')
sub.head()

OK: 4909 строк
Score range: [0.0007, 0.9995]
Score mean: 0.5000


,cookie_id,score
0,ck_315fb710a0e371e7,0.079470
1,ck_a76ee3b3e3e522fd,0.889572
2,ck_94c9a4d382689e82,0.758045
3,ck_8eaf9509ad9462a0,0.018859
4,ck_9a88a5a989cb5bc6,0.369654
